# Step 4: Baseline experiment
**CS201 Project: Time-aware TSP around elephant crossing zones**

What this notebook does:
1. Loads the team simulator (`simulator.py` at the repo root), the 161 towns and the elephant zones.
2. Runs quick tests to check the simulator.
3. **Baseline**: classic Nearest Neighbour, which ignores the zones and the clock.
4. Runs the baseline through the simulator and counts the unsafe legs (motivation slide: classic TSP fails here).
5. Repeats it for every start hour (0:00 to 23:00) to show the problem depends on the time of day.
6. Saves a map, a chart and CSVs to `output/`.

All the simulator code (the clock, the zone checks, the roads, Nearest Neighbour) lives in `simulator.py`, so the baseline, Approach A and Approach B are all checked by exactly the same code.

## 1. Load the simulator and the data

In [ ]:
import os, sys, csv, time
ROOT = ".." if os.path.exists("../simulator.py") else "."     # repo root
sys.path.append(ROOT)
import simulator as sim

IN_DIR = os.path.join(ROOT, "01_data_prep")
ZONES_DIR = os.path.join(ROOT, "data", "zones(augmented data)")
OUT_DIR = os.path.join(ROOT, "02_baseline_simulator", "output")
START_TIME = "08:00"

print(sim.load(IN_DIR, ZONES_DIR, start_town="Ipoh"))
print(f"Speed {sim.SPEED_KMH} km/h, checked every {sim.SAMPLE_KM} km")
for z in sim.zones:
    print(f"  {z['id']:>3}: {z['name']}, on {z['route']}, buffer {z['buffer_km']} km, "
          f"active {z['active'][0]}-{z['active'][1]}")

## 2. Quick tests of the simulator
Gerik → Kampung Jeli drives along Route 4, so it should be safe in the afternoon and hit every Route 4 zone at night. The Route 185 detour should be safe in the afternoon and hit its own zones at night.

In [ ]:
z = sim.zones[0]
assert sim.zone_active(z, sim.to_min("21:00")) and sim.zone_active(z, sim.to_min("03:00"))
assert not sim.zone_active(z, sim.to_min("14:00"))
assert sim.zone_active(z, sim.to_min("21:00") + 1440)       # next day works too

g, j, kb = sim.idx["Gerik"], sim.idx["Kampung Jeli"], sim.idx["Kota Bharu"]
r4_ids = {z["id"] for z in sim.zones if z["route"] == "R4"}
r185_ids = {z["id"] for z in sim.zones if z["route"] == "R185"}
assert not sim.check_leg(g, j, sim.to_min("14:00"))[2], "should be safe in the afternoon"
assert r4_ids <= set(sim.check_leg(g, j, sim.to_min("21:00"))[2]), "should hit every Route 4 zone at night"
assert not sim.check_leg(sim.start, kb, sim.to_min("14:00"), "R185")[2], "detour should be safe in the afternoon"
assert r185_ids <= set(sim.check_leg(sim.start, kb, sim.to_min("21:00"), "R185")[2]), "detour should hit its zones at night"
for clock in ["14:00", "21:00"]:
    arrive, km, hits = sim.check_leg(g, j, sim.to_min(clock))
    print(f"Gerik -> Kampung Jeli leaving {clock}: {km:.0f} km, arrive {sim.to_clock(arrive)}, zones hit: {hits or 'none'}")
print("All simulator tests passed")

## 3. Baseline: classic Nearest Neighbour
`sim.nearest_neighbour()` is the pseudocode from the project brief: always go to the closest unvisited town (by `distances_km.csv`), ignoring the zones and the clock. Any Perak ↔ Kelantan leg is driven on Route 4.

Complexity: O(n²) time, O(n) extra memory on top of the distance matrix.

In [ ]:
t0 = time.perf_counter()
baseline_tour = sim.nearest_neighbour()
runtime_ms = (time.perf_counter() - t0) * 1000
assert len(baseline_tour) == sim.n + 1 and len(set(baseline_tour)) == sim.n
print(f"Tour visits {sim.n} towns, runtime {runtime_ms:.2f} ms")

## 4. Run the baseline through the simulator

In [ ]:
result = sim.simulate(baseline_tour, sim.to_min(START_TIME))
print(f"Start {START_TIME} from {sim.towns[sim.start]['name']}")
print(f"Total distance: {result['total_km']:.1f} km")
print(f"Total trip time: {result['total_min'] / 60:.1f} h  (finish {sim.to_clock(sim.to_min(START_TIME) + result['total_min'])})")
print(f"Unsafe legs: {result['unsafe_legs']} of {len(result['legs'])}")
print()
for k, l in enumerate(result["legs"]):
    if l["route"] != "local" or l["unsafe"]:
        print(f"  leg {k:3d}: {sim.towns[l['from']]['name']} -> {sim.towns[l['to']]['name']} ({l['route']}), "
              f"{sim.to_clock(l['depart'])} to {sim.to_clock(l['arrive'])}, "
              f"{'UNSAFE, zones ' + ', '.join(l['zones_hit']) if l['unsafe'] else 'safe'}")

## 5. Does the start time matter?
The baseline tour is always the same order (it ignores time), but **when** each leg is driven changes with the start time.

In [ ]:
by_hour = []
for h in range(24):
    r = sim.simulate(baseline_tour, h * 60)
    by_hour.append((h, r["unsafe_legs"], sum(len(l["zones_hit"]) for l in r["legs"])))
    print(f"  start {h:02d}:00 -> {by_hour[-1][1]} unsafe legs, {by_hour[-1][2]} active zones driven through")

## 6. Map and chart (for the slides)

In [ ]:
import matplotlib.pyplot as plt
os.makedirs(OUT_DIR, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 8))
for l in result["legs"]:
    p = sim.road_path(l["from"], l["to"], "R4" if l["route"] == "local" else l["route"])
    ax.plot([q[1] for q in p], [q[0] for q in p],
            color="red" if l["unsafe"] else "grey", lw=2 if l["unsafe"] else 0.7)
ax.scatter([t["lon"] for t in sim.towns], [t["lat"] for t in sim.towns], s=8, color="black", zorder=3)
for z in sim.zones:
    ax.plot([p[1] for p in z["points"]], [p[0] for p in z["points"]], color="orange", lw=8, alpha=0.7)
ax.scatter([sim.towns[sim.start]["lon"]], [sim.towns[sim.start]["lat"]], s=80, marker="*", color="blue", zorder=4)
ax.set_title(f"Baseline NN, start {START_TIME}: {result['unsafe_legs']} unsafe legs (red), zones (orange)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.set_aspect("equal")
plt.savefig(os.path.join(OUT_DIR, "baseline_map.png"), dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar([h for h, _, _ in by_hour], [u for _, u, _ in by_hour], color="red")
ax.set_xlabel("Start hour"); ax.set_ylabel("Unsafe legs"); ax.set_xticks(range(0, 24, 2))
ax.set_title("Baseline NN: unsafe legs by start time")
plt.savefig(os.path.join(OUT_DIR, "baseline_unsafe_by_hour.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Save the output files

In [ ]:
with open(os.path.join(OUT_DIR, "baseline_legs.csv"), "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["leg", "from_id", "from", "to_id", "to", "route", "km", "wait_min",
                "depart", "arrive", "unsafe", "zones_hit"])
    for k, l in enumerate(result["legs"]):
        w.writerow([k, l["from"], sim.towns[l["from"]]["name"], l["to"], sim.towns[l["to"]]["name"], l["route"],
                    round(l["km"], 3), l["wait_min"], sim.to_clock(l["depart"]), sim.to_clock(l["arrive"]),
                    l["unsafe"], ";".join(l["zones_hit"])])

with open(os.path.join(OUT_DIR, "baseline_unsafe_by_hour.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["start_hour", "unsafe_legs", "active_zones_driven_through"])
    w.writerows(by_hour)

lines = [
    f"Zones: {', '.join(z['id'] for z in sim.zones)}",
    f"Settings: start {START_TIME} at {sim.towns[sim.start]['name']}, speed {sim.SPEED_KMH} km/h, sample every {sim.SAMPLE_KM} km",
    f"Towns: {sim.n}",
    f"Baseline NN: {result['total_km']:.1f} km, {result['total_min'] / 60:.1f} h, "
    f"{result['unsafe_legs']} unsafe legs, runtime {runtime_ms:.2f} ms",
    "Unsafe legs by start hour: " + ", ".join(f"{h:02d}h={u}" for h, u, _ in by_hour),
]
text = "\n".join(lines)
with open(os.path.join(OUT_DIR, "summary.txt"), "w") as f:
    f.write(text + "\n")
print(text)
print("Saved to", OUT_DIR)